# Análisis de los hábitos de escucha en Spotify

Cuaderno de análisis del proyecto final de Técnicas de Ciencia de Datos. Parte de los dos CSV generados con `TrabajoTecnicasDeCienciaDeDatos.ipynb` y desarrolla el modelo de datos, el enriquecimiento, la compatibilidad entre usuarios, la afinidad con artistas nuevos y el agrupamiento por géneros.

## Orden de ejecución

1. Ejecutar el cuaderno de extracción con la cuenta de cada autor para generar `top_artistas.csv` y `top_artistasJorge.csv`.
2. Ejecutar este cuaderno completo con *Restart & Run All*.

## Privacidad

Los CSV crudos, la caché de enriquecimiento y las tablas con nombres de artistas no se versionan. Solo se publican agregados (`resultados/*.csv` y `resultados/metricas.json`) y las figuras del informe.


In [ ]:
from pathlib import Path
import json

from dotenv import load_dotenv

load_dotenv()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

import spotify_analisis as sa

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})
sns.set_theme(style="whitegrid", context="notebook")

DIR_RES = Path("resultados")
DIR_RES.mkdir(exist_ok=True)
RUTAS = {"pablo": "top_artistas.csv", "jorge": "top_artistasJorge.csv"}


## 1. Carga y validación

In [ ]:
df = sa.cargar_datos(RUTAS)
validacion = sa.validar_datos(df)
display(validacion)
df.head()


## 2. Consolidación y géneros

In [ ]:
generos = sa.explosionar_generos(df)
resumen = sa.resumen_por_usuario_rango(df)
principales = sa.genero_principal(generos)
resumen.to_csv(DIR_RES / "resumen_datos.csv", index=False)
display(resumen)


In [ ]:
filas_div = []
for (usuario, rango), grupo in generos.groupby(["usuario", "rango"], sort=False):
    conteo = grupo.drop_duplicates(["artista", "genero"])["genero"].value_counts()
    filas_div.append({
        "usuario": usuario,
        "rango": rango,
        "n_artistas": grupo["artista"].nunique(),
        "n_generos": len(conteo),
        "entropia": round(sa.entropia_shannon(conteo), 4),
        "cuota_genero_top": round(float(conteo.iloc[0] / conteo.sum()), 4) if len(conteo) else 0.0,
    })
diversidad = pd.DataFrame(filas_div)
display(diversidad)


In [ ]:
usuarios = sorted(df["usuario"].unique())
fig, axes = plt.subplots(1, len(usuarios), figsize=(12, 6))
for ax, usuario in zip(np.atleast_1d(axes), usuarios):
    perfil = sa.perfil_generos(generos, usuario=usuario).head(15).iloc[::-1]
    ax.barh(perfil.index, perfil.values, color="#4c72b0")
    ax.set_title(f"Top 15 géneros · {usuario.capitalize()}")
    ax.set_xlabel("Nº de artistas distintos")
fig.tight_layout()
sa.guardar_figura(fig, "01_top_generos")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(usuarios), figsize=(12, 5))
for ax, usuario in zip(np.atleast_1d(axes), usuarios):
    tabla = generos[generos["usuario"] == usuario].drop_duplicates(["rango", "artista", "genero"])
    conteo = tabla.groupby(["rango", "genero"]).size().unstack(fill_value=0)
    top = conteo.sum().sort_values(ascending=False).head(8).index.tolist()
    resto = [g for g in conteo.columns if g not in top]
    datos = conteo[top].copy()
    datos["otros"] = conteo[resto].sum(axis=1)
    shares = datos.div(datos.sum(axis=1), axis=0)
    shares = shares.reindex([r for r in sa.ORDEN_RANGOS if r in shares.index])
    shares.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
    ax.set_ylabel("Proporción")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.set_title(f"Mezcla de géneros por rango · {usuario.capitalize()}")
    ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
sa.guardar_figura(fig, "02_generos_por_rango")
plt.show()


## 3. Estabilidad temporal

In [ ]:
estabilidad = pd.concat(
    [sa.estabilidad_por_rangos(generos, u) for u in sorted(df["usuario"].unique())],
    ignore_index=True,
)
display(estabilidad)

presencia = (
    df.groupby(["usuario", "artista"])["rango"]
    .nunique()
    .rename("n_rangos")
    .reset_index()
    .groupby(["usuario", "n_rangos"])
    .size()
    .unstack(fill_value=0)
)
display(presencia)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
rangos = list(sa.ORDEN_RANGOS)
cortos = {"ultimas_4_semanas": "4 semanas", "ultimos_6_meses": "6 meses", "varios_anios": "varios años"}
for ax, usuario in zip(axes, sorted(df["usuario"].unique())):
    matriz = pd.DataFrame(np.eye(len(rangos)), index=rangos, columns=rangos)
    sub = estabilidad[estabilidad["usuario"] == usuario]
    for fila in sub.itertuples(index=False):
        matriz.loc[fila.rango_a, fila.rango_b] = fila.jaccard_artistas
        matriz.loc[fila.rango_b, fila.rango_a] = fila.jaccard_artistas
    sns.heatmap(
        matriz,
        annot=True,
        fmt=".2f",
        vmin=0,
        vmax=1,
        cmap="Blues",
        ax=ax,
        xticklabels=[cortos[r] for r in rangos],
        yticklabels=[cortos[r] for r in rangos],
        cbar=False,
    )
    ax.set_title(f"Jaccard de artistas entre rangos · {usuario.capitalize()}")
    ax.set_xlabel("")
    ax.set_ylabel("")
fig.tight_layout()
sa.guardar_figura(fig, "03_estabilidad_rangos")
plt.show()


## 4. Enriquecimiento y valores nulos

In [ ]:
artistas_unicos = sorted(df["artista"].unique())
artistas, similitudes = sa.enriquecer_artistas(artistas_unicos)
artistas = artistas.merge(
    principales.rename("genero_principal"),
    left_on="artista",
    right_index=True,
    how="left",
)
generos_por_artista = df.drop_duplicates("artista").set_index("artista")["generos"]
artistas["generos"] = artistas["artista"].map(generos_por_artista)
nulos = sa.resumen_nulos(artistas)
nulos.to_csv(DIR_RES / "nulos.csv", index=False)
display(nulos)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(nulos["columna"], nulos["pct_nulos"], color="#c44e52")
ax.set_xlabel("% de valores nulos")
ax.set_xlim(0, 100)
ax.set_title("Cobertura de los atributos enriquecidos")
ax.invert_yaxis()
fig.tight_layout()
sa.guardar_figura(fig, "09_nulos")
plt.show()


In [ ]:
artistas, inf_listeners = sa.imputar_popularidad(artistas, "listeners")
artistas, inf_playcount = sa.imputar_popularidad(artistas, "playcount")
artistas, inf_deezer = sa.imputar_popularidad(artistas, "deezer_fans")
informe_imputacion = pd.concat([inf_listeners, inf_playcount, inf_deezer], ignore_index=True)
display(informe_imputacion)

pares = artistas.dropna(subset=["listeners", "deezer_fans"])
rho_deezer, p_deezer = spearmanr(pares["listeners"], pares["deezer_fans"])
print(f"Spearman listeners (Last.fm) vs fans (Deezer): rho={rho_deezer:.3f}, p={p_deezer:.3g}, n={len(pares)}")


## 5. Compatibilidad entre usuarios

In [ ]:
compat = sa.tabla_compatibilidad(generos)
compat.to_csv(DIR_RES / "compatibilidad.csv", index=False)
display(compat)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(compat))
ancho = 0.38
ax.bar(x - ancho / 2, compat["jaccard_artistas"], ancho, label="Jaccard (artistas)")
ax.bar(x + ancho / 2, compat["coseno_generos"], ancho, label="Coseno (géneros)")
ax.set_xticks(x)
ax.set_xticklabels([r.replace("_", " ") for r in compat["rango"]])
ax.set_ylim(0, 1)
ax.set_ylabel("Similitud")
ax.set_title("Compatibilidad musical entre los dos usuarios")
ax.legend()
fig.tight_layout()
sa.guardar_figura(fig, "04_compatibilidad")
plt.show()


## 6. Popularidad frente a la posición en el ranking

In [ ]:
datos_pop = sa.unir_enriquecimiento(df, artistas)
tabla_spearman = sa.spearman_por_grupo(datos_pop)
tabla_spearman.to_csv(DIR_RES / "popularidad_spearman.csv", index=False)
display(tabla_spearman)

fig, ax = plt.subplots(figsize=(8, 5))
for usuario, color in zip(sorted(df["usuario"].unique()), ["#4c72b0", "#dd8452"]):
    sub = datos_pop[
        (datos_pop["usuario"] == usuario)
        & (datos_pop["rango"] == "varios_anios")
    ].dropna(subset=["listeners"])
    ax.scatter(sub["posicion"], sub["listeners"], alpha=0.65, label=usuario.capitalize(), color=color)
ax.set_yscale("log")
ax.set_xlabel("Posición en el ranking")
ax.set_ylabel("Oyentes en Last.fm (escala logarítmica)")
ax.set_title("Popularidad del artista frente a su posición (largo plazo)")
ax.legend()
fig.tight_layout()
sa.guardar_figura(fig, "05_popularidad_rank")
plt.show()


## 7. Afinidad usuario–artista

In [ ]:
candidatos = sa.generar_candidatos(df, similitudes)
afinidad = sa.calcular_afinidad(df, similitudes, candidatos)
usuarios = sorted(df["usuario"].unique())
afinidad["media"] = afinidad[usuarios].mean(axis=1)
afinidad = afinidad.sort_values("media", ascending=False).reset_index(drop=True)
afinidad.to_csv(DIR_RES / "afinidad_resumen.csv", index=False)
display(afinidad)

loo = sa.evaluar_afinidad_loo(df, similitudes, candidatos)
display(loo)


In [ ]:
top = afinidad.head(10).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(top))
alto = 0.38
for desplazamiento, usuario, color in zip(
    (-alto / 2, alto / 2), usuarios, ["#4c72b0", "#dd8452"]
):
    ax.barh(y + desplazamiento, top[usuario], alto, label=usuario.capitalize(), color=color)
ax.set_yticks(y)
ax.set_yticklabels([f"Candidato {i}" for i in range(1, len(top) + 1)])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("Afinidad estimada")
ax.set_title("Afinidad estimada con artistas nuevos (candidatos anonimizados)")
ax.legend()
fig.tight_layout()
sa.guardar_figura(fig, "06_afinidad")
plt.show()


## 8. Agrupamiento de artistas por género

In [ ]:
X_df, info_clusters = sa.matriz_dummy_artistas(generos)
print(info_clusters)
tabla_k, k_elegido = sa.elegir_k(X_df.values)
display(tabla_k)
print("k elegido:", k_elegido)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(tabla_k["k"], tabla_k["inercia"], marker="o", color="#4c72b0", label="Inercia")
ax1.set_xlabel("Número de clusters (k)")
ax1.set_ylabel("Inercia", color="#4c72b0")
ax2 = ax1.twinx()
ax2.plot(tabla_k["k"], tabla_k["silhouette"], marker="s", color="#c44e52", label="Silhouette")
ax2.set_ylabel("Silhouette", color="#c44e52")
ax1.axvline(k_elegido, linestyle="--", color="grey")
ax1.set_title("Selección del número de clusters")
fig.tight_layout()
sa.guardar_figura(fig, "08_seleccion_k")
plt.show()


In [ ]:
etiquetas, modelo = sa.ajustar_clusters(X_df.values, k_elegido)
perfil_clusters = sa.etiquetar_clusters(X_df, etiquetas)
perfil_clusters.to_csv(DIR_RES / "clusters_resumen.csv", index=False)
display(perfil_clusters)

coords, varianza = sa.proyectar_pca(X_df.values)
fig, ax = plt.subplots(figsize=(7, 5))
paleta = sns.color_palette("tab10", n_colors=k_elegido)
for cluster in range(k_elegido):
    mascara = etiquetas == cluster
    ax.scatter(coords[mascara, 0], coords[mascara, 1], s=28, alpha=0.7, color=paleta[cluster], label=f"Cluster {cluster}")
centroides = np.array([coords[etiquetas == c].mean(axis=0) for c in range(k_elegido)])
ax.scatter(centroides[:, 0], centroides[:, 1], marker="X", s=160, c="black", label="Centroides")
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
ax.set_title(f"Proyección PCA de los clusters (varianza explicada: {varianza:.1%})")
ax.legend(fontsize=8)
fig.tight_layout()
sa.guardar_figura(fig, "07_clustering_pca")
plt.show()


In [ ]:
generos_frecuentes = X_df.mean(axis=0).sort_values(ascending=False).head(12).index
prevalencia = pd.DataFrame(
    {c: X_df.loc[etiquetas == c, generos_frecuentes].mean(axis=0) for c in range(k_elegido)}
).T
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(prevalencia, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax)
ax.set_xlabel("Género")
ax.set_ylabel("Cluster")
ax.set_title("Prevalencia de los géneros más frecuentes por cluster")
fig.tight_layout()
sa.guardar_figura(fig, "07b_perfil_clusters")
plt.show()

artista_usuario = df.drop_duplicates(["artista", "usuario"])[["artista", "usuario"]]
clusters_df = pd.DataFrame({"artista": X_df.index, "cluster": etiquetas}).merge(
    artista_usuario, on="artista", how="left"
)
display(pd.crosstab(clusters_df["cluster"], clusters_df["usuario"]))


## 9. Síntesis y exportación de métricas

In [ ]:
afinidad_anon = afinidad.head(10).copy()
afinidad_anon.insert(0, "candidato", [f"C{i:02d}" for i in range(1, len(afinidad_anon) + 1)])

metricas = {
    "meta": {
        "fecha_ejecucion": pd.Timestamp.now().strftime("%Y-%m-%d"),
        "usuarios": usuarios,
        "n_artistas_unicos": int(df["artista"].nunique()),
    },
    "validacion": validacion.to_dict(orient="records"),
    "diversidad": diversidad.to_dict(orient="records"),
    "estabilidad": estabilidad.to_dict(orient="records"),
    "compatibilidad": compat.to_dict(orient="records"),
    "popularidad": tabla_spearman.to_dict(orient="records"),
    "afinidad": {
        "loo": loo.to_dict(orient="records"),
        "top10": afinidad_anon[["candidato", *usuarios]].to_dict(orient="records"),
    },
    "clustering": {
        "info": info_clusters,
        "k": int(k_elegido),
        "seleccion_k": tabla_k.to_dict(orient="records"),
        "perfiles": perfil_clusters.to_dict(orient="records"),
        "varianza_pca": round(varianza, 4),
    },
    "nulos": nulos.to_dict(orient="records"),
    "imputacion": informe_imputacion.to_dict(orient="records"),
    "deezer_vs_lastfm": {
        "rho": round(float(rho_deezer), 4),
        "p": float(p_deezer),
        "n": int(len(pares)),
    },
}
with open(DIR_RES / "metricas.json", "w", encoding="utf-8") as fichero:
    json.dump(metricas, fichero, ensure_ascii=False, indent=2)
print("Claves exportadas:", list(metricas))


## 10. Conclusiones preliminares

- La cobertura de géneros y de metadatos de Last.fm/Deezer determina qué artistas pueden analizarse; los huecos se documentan y no se imputan cuando no existe un sustituto razonable.
- La compatibilidad entre los dos usuarios se mide con dos métricas complementarias (conjuntos de artistas y frecuencias de género), por rango y en conjunto.
- La afinidad usuario–artista es una estimación descriptiva basada en similitud de contenido, no una predicción supervisada.
- El clustering agrupa artistas por género; los grupos deben interpretarse como descriptores del perfil musical, no como categorías naturales validadas.
